# **NeuroScan: Brain Tumor MRI Classification with Controlled Leakage Analysis**

### CS 475/675 — Final Project Report

This notebook presents NeuroScan, a two-stage deep learning pipeline for classifying brain MRI scans into eight tumor categories with conditional glioma subtyping. Beyond the classification system itself, a central methodological contribution is a controlled investigation of evaluation contamination in widely-used brain tumor MRI benchmarks, where patient-level data leakage inflates reported accuracy by up to 11.6 percentage points for specific classes.

# Project Name: NeuroScan
Project mentor: N/A

Solo Project — kshay3@jh.edu — Kiran Shay

Link to git repo: [github.com/kiranshay/brain-tumor-classifier](https://github.com/kiranshay/brain-tumor-classifier)

Live demo: [kiranshay.github.io/brain-tumor-classifier](https://kiranshay.github.io/brain-tumor-classifier)

# Outline and Deliverables

### Completed Deliverables

**Must accomplish:**

1. **8-class Stage 1 classifier trained and evaluated** — EfficientNet-B0 on the combined dataset, reporting per-class precision/recall/F1, confusion matrix, and Wilson 95% confidence intervals. Covered in [Results](#results) and [Experimental Setup](#experimental-setup).

2. **Clean dataset with patient-aware splits** — Built via `build_clean_dataset.py` using inferred patient IDs and `GroupShuffleSplit` on data44, with preserved published splits on data1/data2. Quantified leakage inflation at +2.52 pp overall, +11.6 pp on d1 glioma. Covered in [Pre-processing](#pre-processing) and [Results](#results).

3. **Wilson 95% confidence intervals for all per-class metrics** — Computed via `wilson_intervals.py`, revealing that rare-class point estimates (e.g., papilloma recall 0.657) carry 30-percentage-point intervals at n=35. Covered in [Results](#results).

**Expected to accomplish:**

4. **Stage 2 glioma subtype classifier** — 4-class EfficientNet-B0 (astrocytoma, ependymoma, glioblastoma, oligodendroglioma) integrated into a conditional cascade: Stage 2 fires only when Stage 1 predicts glioma. Covered in [Methods](#methods).

5. **Grad-CAM interpretability** — Implemented via hooks on the final convolutional block, generating heatmaps overlaid on input MRIs. Available on-demand through the deployed web app. Covered in [Methods](#methods).

6. **Source confound analysis** — Analyzed whether data44-only rare classes achieve high accuracy via scanner signature recognition rather than tumor morphology. Covered in [Results](#results).

**Would like to accomplish:**

7. **Full-stack web deployment** — FastAPI backend on Render, vanilla JS frontend on GitHub Pages, Supabase Postgres for prediction history and statistics. Live at the demo link above. Covered in [Methods](#methods).

### Uncompleted Deliverables

8. **Class-weighted loss / weighted random sampling** — Not implemented. The clean training pipeline (`train_clean.py`) uses unweighted `CrossEntropyLoss`. Addressing class imbalance remains future work; the Wilson confidence intervals partially compensate by quantifying the resulting uncertainty on rare classes.

9. **Standalone contamination research contribution** — The controlled leakage experiment is complete and documented, but was not developed into a separate paper submission. The per-source analysis showing that leakage selectively inflates the classes with real distributional problems is the key finding that would anchor such a contribution.

# Preliminaries

## What problem were you trying to solve or understand?

Brain tumor diagnosis from MRI scans requires expert radiological interpretation that is time-consuming and subject to inter-observer variability. Automated classification could serve as a screening aid, flagging suspicious scans for priority review. The task is a multi-class image classification problem — structurally similar to the image classification problems we studied in lecture (CIFAR-10, ImageNet), but with domain-specific complications: class imbalance, multiple imaging sequences (T1, T2, T1C+), and the risk of data leakage from multi-slice patients.

**What makes this problem unique** is the evaluation methodology challenge. Most published results on these Kaggle brain tumor benchmarks use image-level train/test splits that ignore patient identity. When a patient contributes multiple MRI slices, random splitting places near-duplicate images in both sets, inflating test accuracy. This project addresses that gap with a controlled experiment isolating the effect of leakage — not just building a classifier, but measuring how much the dominant evaluation pattern overstates performance.

**Ethical implications** are significant. A classifier deployed in a clinical screening context that reports 99% accuracy based on leaked evaluation would give false confidence. The +11.6 pp inflation we measured on d1 glioma recall means a practitioner using the standard Kaggle split would see ~98% recall and have no reason to investigate, while the honest evaluation reveals 86.3% — a gap that matters when the downstream decision is whether to order a biopsy. The model's uncertainty on rare classes (papilloma recall CI spans [0.49, 0.79]) further underscores that point estimates alone are insufficient for clinical decision-making.

## Datasets

Three publicly available Kaggle datasets were combined, totaling over 15,000 MRI images:

| Source | Kaggle Slug | Classes | Images | Split Provided? |
|--------|------------|---------|--------|-----------------|
| **Brain Tumor MRI (data1)** | `masoudnickparvar/brain-tumor-mri-dataset` | 4 (glioma, meningioma, no tumor, pituitary) | ~7,023 | Yes (Training/Testing folders) |
| **BRISC 2025 (data2)** | `briscdataset/brisc2025` | 4 (same as above) | ~6,000 | Yes (train/test folders) |
| **Brain Tumor 44-class (data44)** | `fernando2rad/brain-tumor-mri-images-44c` | 44 fine-grained types across T1, T2, T1C+ sequences | ~2,500+ | No |

**Data1** is the most commonly used brain tumor MRI benchmark on Kaggle. It provides clean 4-class labels with a pre-defined train/test split, but likely contains multiple slices per patient (inferred from near-duplicate images across splits).

**Data2 (BRISC)** adds axial, sagittal, and coronal orientations for the same 4 classes, increasing diversity. Its multi-orientation nature means it almost certainly has patient-level overlap within its published split.

**Data44** provides fine-grained tumor types organized as `<TumorName> <Sequence>` folders (e.g., "Astrocitoma T1", "Meningioma T1C+"). This is the sole source for the four rare classes (schwannoma, neurocytoma, carcinoma, papilloma) and all glioma subtype labels. It has no pre-defined split, so we constructed one.

The 8-class Stage 1 model trains on all three datasets combined. The trustworthy 4-class model and the leaky control use only data1 + data2 (to keep the leakage experiment clean). The glioma subtype Stage 2 model uses data44 only.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown

RESULTS = Path("results")

# Load run summaries for all three experiments
for run_name in ["trustworthy_4class", "extended_8class", "leaky_4class"]:
    summary_path = RESULTS / run_name / "run_summary.json"
    with open(summary_path) as f:
        summary = json.load(f)
    print(f"=== {run_name} ===")
    print(f"  Classes: {summary['num_classes']} — {summary.get('class_names', 'N/A')}")
    ds = summary.get("dataset_sizes", {})
    print(f"  Train: {ds.get('train', 'N/A')}  Val: {ds.get('val', 'N/A')}  Test: {ds.get('test', 'N/A')}")
    print(f"  Test accuracy: {summary['test_accuracy']:.4f}")
    print(f"  Test loss: {summary['test_loss']:.4f}")
    print(f"  Best epoch: {summary['best_epoch']} (val acc: {summary['best_val_accuracy']:.4f})")
    print(f"  Training time: {summary['training_time_seconds']:.0f}s ({summary['training_time_seconds']/60:.1f} min)")
    print()

## Pre-processing

**Features:** Each input is a single brain MRI slice (grayscale or pseudo-color depending on the source dataset). All features are continuous pixel intensities — there are no hand-crafted features. The model learns features end-to-end via transfer learning from ImageNet.

**MRI-specific border cropping:** Brain MRI scans often have large black bezels from the scanner. A preprocessing step converts to grayscale, masks rows/columns whose mean intensity exceeds a threshold of 15, crops to the bounding box of the mask, and pads 5px. This removes scanner-specific borders that could serve as spurious features.

**Resizing and normalization:** Images are resized to 224x224 (matching EfficientNet-B0's expected input) and normalized with ImageNet mean (`[0.485, 0.456, 0.406]`) and std (`[0.229, 0.224, 0.225]`).

**Data augmentation (training only):**
- Resize to 256, then RandomCrop to 224
- Random horizontal and vertical flips
- Random rotation (±20°)
- Random affine (translate 0.1, scale 0.9–1.1)
- Color jitter (brightness 0.3, contrast 0.3, saturation 0.2)
- Random grayscale (p=0.1)
- Random erasing (p=0.2)

**Class balance:** The datasets are imbalanced. In the 8-class build, glioma (sourced from all three datasets) has several times more images than papilloma or carcinoma (sourced from data44 only, n≈35 test images each). No class weighting, oversampling, or balanced loss was applied — instead, Wilson confidence intervals quantify the resulting uncertainty on rare classes.

**Split strategy (the core methodological contribution):**

For the **trustworthy builds**, we preserved the published train/test splits from data1 and data2, and carved a 15% file-level validation set from training data. For data44 (no published split), we attempted patient-level grouping via filename heuristics (digit runs, leading filename tokens) and used these groups for splitting; when heuristics were unreliable, the builder falls back to image-level split with a documented warning.

For the **leaky control**, we pooled all images from the trustworthy build and re-split at the image level with `random.shuffle(seed=42)`, maintaining identical train/val/test ratios. Everything else — architecture, optimizer, augmentation, seed — was held constant. The only variable is the split strategy.

In [ ]:
# Show class distributions for each build
for run_name in ["trustworthy_4class", "extended_8class", "leaky_4class"]:
    report_path = RESULTS / run_name / "test_report.json"
    with open(report_path) as f:
        report = json.load(f)
    
    print(f"=== {run_name}: test set class distribution ===")
    classes = sorted([k for k in report if k not in ("accuracy", "macro avg", "weighted avg")])
    for cls in classes:
        m = report[cls]
        print(f"  {cls:20s}  n={int(m['support']):>5d}   P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1-score']:.3f}")
    print(f"  {'macro avg':20s}  P={report['macro avg']['precision']:.3f}  R={report['macro avg']['recall']:.3f}  F1={report['macro avg']['f1-score']:.3f}")
    print(f"  {'weighted avg':20s}  P={report['weighted avg']['precision']:.3f}  R={report['weighted avg']['recall']:.3f}  F1={report['weighted avg']['f1-score']:.3f}")
    print()

In [ ]:
# Visualize class distribution across the three builds
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, run_name in zip(axes, ["trustworthy_4class", "extended_8class", "leaky_4class"]):
    with open(RESULTS / run_name / "test_report.json") as f:
        report = json.load(f)
    
    classes = sorted([k for k in report if k not in ("accuracy", "macro avg", "weighted avg")])
    supports = [int(report[c]["support"]) for c in classes]
    
    bars = ax.barh(classes, supports, color=plt.cm.Set2(np.linspace(0, 1, len(classes))))
    ax.set_xlabel("Test set size")
    ax.set_title(run_name.replace("_", " ").title())
    for bar, s in zip(bars, supports):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, str(s), va="center", fontsize=9)

plt.tight_layout()
plt.suptitle("Test Set Class Distribution Across Builds", y=1.02, fontsize=14)
plt.show()

# Models and Evaluation

## Experimental Setup

**Evaluation metrics:** We report per-class precision, recall, and F1-score, along with macro and weighted averages. For a medical imaging task, recall (sensitivity) matters most — a missed tumor is more dangerous than a false alarm. We use macro F1 to avoid the weighted average masking poor performance on rare classes. Wilson 95% confidence intervals on per-class precision and recall are critical for rare classes where n < 70; standard point estimates give a misleading sense of certainty.

**Loss function:** Cross-entropy loss (`CrossEntropyLoss`) throughout. This is the standard choice for multi-class classification with softmax outputs. We did not try focal loss or class-weighted cross-entropy — the Wilson intervals approach was our strategy for handling imbalance at evaluation time rather than training time.

**Train/val/test split:** The trustworthy builds use a proper three-way split: published train/test splits from data1 and data2, with a 15% validation carve-out from the training data. The validation set is used for early stopping (best checkpoint saved on `val_loss`), and the test set is touched only once for final evaluation. This is in contrast to the earlier V1–V4 training runs where the test set was used as the validation set — a methodological flaw corrected in the `train_clean.py` pipeline.

**Controlled leakage experiment:** To isolate the effect of split strategy on reported performance, we constructed a matched pair of runs:
- **Trustworthy:** Published splits preserved, validation carved from train
- **Leaky:** All images pooled and re-split at image level with random shuffle

Everything else held constant (architecture, optimizer, augmentation, RNG seed, dataset sizes within ±3 images). The delta between the two runs is causally attributable to the split strategy alone.

In [ ]:
# Key training configuration (from train_clean.py)
# All three runs used identical hyperparameters — only the dataset/split differed

config = {
    "Architecture": "EfficientNet-B0 (torchvision, pretrained=ImageNet)",
    "Classifier head": "Dropout(0.3) → Linear(1280, num_classes)",
    "Optimizer": "AdamW(lr=5e-5, weight_decay=0.01)",
    "LR Schedule": "CosineAnnealingLR(T_max=20)",
    "Loss": "CrossEntropyLoss (unweighted)",
    "Epochs": 20,
    "Batch size": 32,
    "Best checkpoint": "Saved on lowest val_loss",
    "RNG seed": "42 (random, numpy, torch, cuda, cudnn deterministic)",
    "GPU": "Tesla T4 (Google Colab)",
}

for k, v in config.items():
    print(f"  {k:25s}: {v}")

## Baselines

**Baseline 1: Earlier 4-class models (progressive unfreezing).** Before developing the clean training pipeline, we trained a series of 4-class models on data1 alone using the approach from lecture — progressive unfreezing:

| Phase | Strategy | Epochs | Trainable params | Test accuracy |
|-------|----------|--------|-----------------|---------------|
| 1 | Head only (freeze backbone) | 15 | 5,124 | 85.6% |
| 2 | Last 2 blocks + head | 10 | partial | 92.6% |
| 3 | Full fine-tune (AdamW, cosine LR) | 20 | ~4M | 95.4% |

This progression from 85.6% → 92.6% → 95.4% demonstrates the value of progressive unfreezing — a concept from transfer learning discussed in lecture. However, these runs used the test set as the validation set and had no held-out evaluation, so the 95.4% figure is not directly comparable to the trustworthy build's 96.4%.

**Baseline 2: Random and majority-class baselines.** For 4-class balanced data, random guessing achieves 25% accuracy. Majority-class (meningioma, n=706/2300) achieves 30.7%. The trustworthy model's 96.4% represents a 66+ percentage point improvement over majority class.

**Related work context:** Published results on the `masoudnickparvar` dataset on Kaggle typically report 95–99% accuracy, but virtually all use image-level random splits. Our controlled experiment shows this evaluation methodology inflates accuracy by 2.5 pp overall and up to 11.6 pp on the hardest class. The trustworthy evaluation at 96.4% is therefore a more honest benchmark than most published numbers on this dataset family.

## Methods

### Architecture: EfficientNet-B0 with transfer learning

We chose EfficientNet-B0 as the backbone because it achieves strong ImageNet accuracy (~77%) with only ~5.3M parameters — efficient enough to deploy on Render's free CPU tier with acceptable inference latency. The pretrained ImageNet features transfer well to medical imaging because low-level features (edges, textures) generalize across domains, while the fine-tuning adapts higher-level features to tumor morphology.

The classifier head is replaced with `Dropout(p) → Linear(1280, num_classes)`. We use `p=0.3` for Stage 1 (8 classes) and `p=0.4` for Stage 2 (4 glioma subtypes, smaller dataset). The entire network is fine-tuned end-to-end — after experimenting with progressive unfreezing in the baseline, we found that full fine-tuning with a small learning rate (5e-5) and weight decay (0.01) performed comparably while being simpler.

### Two-stage cascade

Stage 1 classifies into 8 categories. When it predicts "glioma," Stage 2 conditionally fires to subtype the glioma into astrocytoma, ependymoma, glioblastoma, or oligodendroglioma. Both stages run the same preprocessing pipeline and use the same architecture, just with different classifier heads and weights. This cascade design avoids the combinatorial explosion of training a single 12-class model (8 + 4 subtypes) where the subtype classes would be severely underrepresented.

### Grad-CAM interpretability

We implemented Grad-CAM (Gradient-weighted Class Activation Mapping) by registering forward and backward hooks on the final convolutional block of EfficientNet-B0. The procedure: (1) forward pass with `requires_grad=True`, (2) backpropagate the score of the predicted class, (3) compute channel weights as the mean gradient over spatial dimensions, (4) weighted sum of activations → ReLU → normalize to [0,1], (5) overlay on the input image. This provides visual evidence of which brain regions the model attends to for each prediction.

### Training pipeline

The `train_clean.py` script implements best practices that the earlier notebook-based training lacked:
- **Full RNG seeding:** `random`, `numpy`, `torch.manual_seed`, `torch.cuda.manual_seed_all`, `cudnn.deterministic=True`
- **Proper three-way split:** Train/val/test with no leakage between them
- **Best-checkpoint saving:** Saves the model with lowest validation loss, not just the final epoch
- **Per-epoch metric logging:** Writes `metrics.jsonl` for post-hoc analysis
- **Label map persistence:** Saves `label_map.json` to guarantee class index consistency between training and inference

### Deployment

The full system is deployed as a web application: FastAPI backend on Render (CPU-only, ~30s cold start), vanilla JavaScript frontend on GitHub Pages, and Supabase Postgres for persistence. The frontend has a retry loop that polls `/health` for up to 60s to handle cold starts. Predictions are stored with full confidence vectors, inference time, and optional subtype results.

In [ ]:
# Training code lives in training/train_clean.py — see the repo for the full script.
# Here we display the core training loop structure:

training_code = """
# From training/train_clean.py (abbreviated)

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(1280, num_classes),
)

optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()

best_val_loss = float("inf")
for epoch in range(20):
    # Train
    model.train()
    for images, labels in train_loader:
        outputs = model(images.to(device))
        loss = criterion(outputs, labels.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()
    
    # Validate
    model.eval()
    val_loss = evaluate(model, val_loader)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_path)
    
    # Log metrics to metrics.jsonl

# Final evaluation on held-out test set (touched only once)
model.load_state_dict(torch.load(best_path))
test_acc, test_loss, predictions = evaluate(model, test_loader, return_predictions=True)
"""
print(training_code)

In [ ]:
# Display training curves for all three runs
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, run_name in zip(axes, ["trustworthy_4class", "extended_8class", "leaky_4class"]):
    metrics_path = RESULTS / run_name / "metrics.jsonl"
    epochs_data = []
    with open(metrics_path) as f:
        for line in f:
            epochs_data.append(json.loads(line))
    
    epochs = [d["epoch"] for d in epochs_data]
    train_loss = [d["train_loss"] for d in epochs_data]
    val_loss = [d["val_loss"] for d in epochs_data]
    train_acc = [d["train_accuracy"] for d in epochs_data]
    val_acc = [d["val_accuracy"] for d in epochs_data]
    
    ax.plot(epochs, train_loss, "b-", label="Train loss", alpha=0.8)
    ax.plot(epochs, val_loss, "r-", label="Val loss", alpha=0.8)
    ax2 = ax.twinx()
    ax2.plot(epochs, train_acc, "b--", label="Train acc", alpha=0.5)
    ax2.plot(epochs, val_acc, "r--", label="Val acc", alpha=0.5)
    ax2.set_ylim(0.8, 1.0)
    ax2.set_ylabel("Accuracy")
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(run_name.replace("_", " ").title())
    ax.legend(loc="upper left", fontsize=8)
    ax2.legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.suptitle("Training Curves: Loss and Accuracy Over Epochs", y=1.02, fontsize=14)
plt.show()

## Results

### Headline comparison

| Build | Test Accuracy | Test Loss | Macro F1 | Weighted F1 |
|-------|--------------|-----------|----------|-------------|
| **Trustworthy 4-class** | 96.43% | 0.179 | 0.966 | 0.964 |
| **Extended 8-class** | 94.76% | 0.174 | 0.909 | 0.948 |
| **Leaky 4-class** | 98.96% | 0.033 | 0.989 | 0.990 |

The 2.5 percentage point gap between trustworthy (96.43%) and leaky (98.96%) is entirely attributable to the split strategy. This is the cleanest causal estimate of leakage inflation obtainable on this benchmark family without institutional patient-level data.

### The leakage inflation is not uniform

The most surprising result is *where* the inflation concentrates. The per-source per-class breakdown reveals:

| (Source, Class) | Trustworthy Recall | Leaky Recall | Delta |
|-----------------|-------------------|-------------|-------|
| **d1 glioma** | **0.863** | **0.979** | **+11.6 pp** |
| d2 glioma | 0.992 | 1.000 | +0.8 pp |
| d1 meningioma | 0.985 | 0.997 | +1.2 pp |
| d1 no_tumor | 1.000 | 0.994 | -0.6 pp |
| d1 pituitary | 0.965 | 0.970 | +0.5 pp |

The +11.6 pp boost on d1 glioma accounts for the bulk of the overall delta. Non-glioma classes move by less than ±1.3 pp. **Random image-level split evaluation does not inflate test accuracy uniformly — it selectively inflates the classes that have real distributional problems**, masking exactly the failure modes that an honest evaluation would surface.

A practitioner using the dominant Kaggle split pattern would see d1 glioma recall at ~98% and have no reason to investigate further. The published-split evaluation reveals 86.3% — a finding with clinical relevance.

### Source confound on rare classes

The four data44-only classes (carcinoma, neurocytoma, papilloma, schwannoma) cannot be fully trusted as generalization signals because they exist in only one source dataset. The model could partially solve these classes by recognizing data44's scanner/contrast/file-format signature rather than tumor morphology. Papilloma's low recall (0.657) and wide Wilson CI ([0.49, 0.79]) suggest the model struggles most with the smallest, single-source class.

### Wilson confidence intervals reveal hidden uncertainty

Point estimates on rare classes are misleading. The Wilson 95% CIs for the extended 8-class model:

| Class | n | Recall | 95% CI |
|-------|---|--------|--------|
| papilloma | 35 | 0.657 | [0.492, 0.792] |
| carcinoma | 37 | 0.919 | [0.787, 0.972] |
| neurocytoma | 68 | 0.941 | [0.858, 0.977] |
| schwannoma | 69 | 0.884 | [0.788, 0.940] |
| glioma | 836 | 0.925 | [0.906, 0.941] |

Papilloma's recall could plausibly be as low as 49% or as high as 79% — a 30-point interval that a bare "65.7% recall" does not communicate.

In [ ]:
# Display confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, run_name in zip(axes, ["trustworthy_4class", "extended_8class", "leaky_4class"]):
    img = plt.imread(str(RESULTS / run_name / "confusion_matrix.png"))
    ax.imshow(img)
    ax.set_title(run_name.replace("_", " ").title(), fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.suptitle("Confusion Matrices", y=1.02, fontsize=14)
plt.show()

In [ ]:
# Wilson confidence intervals visualization for the extended 8-class model
wilson_df = pd.read_csv(RESULTS / "extended_8class" / "wilson_intervals.csv")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Recall with CIs
classes = wilson_df["class"].values
recall = wilson_df["recall"].values
recall_lo = wilson_df["recall_ci_lo"].values
recall_hi = wilson_df["recall_ci_hi"].values
n_true = wilson_df["n_true"].values

colors = ["#e74c3c" if n < 100 else "#3498db" for n in n_true]
y_pos = range(len(classes))

ax1.barh(y_pos, recall, color=colors, alpha=0.7, height=0.6)
ax1.errorbar(recall, y_pos, xerr=[recall - recall_lo, recall_hi - recall], fmt="none", ecolor="black", capsize=4)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([f"{c} (n={n})" for c, n in zip(classes, n_true)])
ax1.set_xlabel("Recall")
ax1.set_title("Per-Class Recall with Wilson 95% CI\n(red = n < 100, wide uncertainty)")
ax1.set_xlim(0.4, 1.05)
ax1.axvline(x=0.9, color="gray", linestyle="--", alpha=0.5, label="0.9 threshold")
ax1.legend()

# Precision with CIs
precision = wilson_df["precision"].values
prec_lo = wilson_df["precision_ci_lo"].values
prec_hi = wilson_df["precision_ci_hi"].values
n_pred = wilson_df["n_pred"].values

colors2 = ["#e74c3c" if n < 100 else "#3498db" for n in n_pred]
ax2.barh(y_pos, precision, color=colors2, alpha=0.7, height=0.6)
ax2.errorbar(precision, y_pos, xerr=[precision - prec_lo, prec_hi - precision], fmt="none", ecolor="black", capsize=4)
ax2.set_yticks(y_pos)
ax2.set_yticklabels([f"{c} (n={n})" for c, n in zip(classes, n_pred)])
ax2.set_xlabel("Precision")
ax2.set_title("Per-Class Precision with Wilson 95% CI")
ax2.set_xlim(0.4, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side comparison: trustworthy vs leaky recall by class
fig, ax = plt.subplots(figsize=(10, 5))

with open(RESULTS / "trustworthy_4class" / "test_report.json") as f:
    trust_report = json.load(f)
with open(RESULTS / "leaky_4class" / "test_report.json") as f:
    leak_report = json.load(f)

classes_4 = sorted([k for k in trust_report if k not in ("accuracy", "macro avg", "weighted avg")])
trust_recall = [trust_report[c]["recall"] for c in classes_4]
leak_recall = [leak_report[c]["recall"] for c in classes_4]

x = np.arange(len(classes_4))
width = 0.35

bars1 = ax.bar(x - width/2, trust_recall, width, label="Trustworthy (published splits)", color="#3498db", alpha=0.8)
bars2 = ax.bar(x + width/2, leak_recall, width, label="Leaky (image-level shuffle)", color="#e74c3c", alpha=0.8)

for i, (t, l) in enumerate(zip(trust_recall, leak_recall)):
    delta = l - t
    ax.annotate(f"+{delta:.1%}" if delta > 0 else f"{delta:.1%}",
                xy=(i + width/2, l), xytext=(0, 8),
                textcoords="offset points", ha="center", fontsize=9, fontweight="bold",
                color="red" if abs(delta) > 0.02 else "gray")

ax.set_ylabel("Recall")
ax.set_title("Leakage Inflation: Per-Class Recall (Trustworthy vs Leaky)")
ax.set_xticks(x)
ax.set_xticklabels(classes_4)
ax.set_ylim(0.8, 1.02)
ax.legend()
plt.tight_layout()
plt.show()

print("The +11.6 pp inflation on glioma recall accounts for the bulk of the +2.52 pp overall accuracy gap.")

# Discussion

## What I've learned

**Transfer learning is remarkably effective for medical imaging, but evaluation methodology matters more than architecture choice.** The jump from a randomly initialized model to an ImageNet-pretrained EfficientNet-B0 is enormous — the pretrained features give you edges, textures, and shapes for free, and fine-tuning adapts them to tumor morphology. But the difference between 96.4% (trustworthy) and 99.0% (leaky) is entirely an artifact of how you split the data. The architecture contributes real performance; the evaluation methodology determines whether you can measure it honestly.

**The most important concept from lecture was train/val/test discipline.** Early in the project, I used the test set as the validation set and had no held-out evaluation — the same mistake I would have caught in a homework. Building `train_clean.py` with proper three-way splits, best-checkpoint saving on validation loss, and a test set touched only once was the single biggest methodological improvement. The progressive unfreezing experiments from lecture (head-only → partial → full) were also directly useful for building intuition about how much to unfreeze.

**The most surprising finding was that leakage inflation is not uniform.** I expected image-level random splits to inflate accuracy across all classes roughly equally. Instead, the +11.6 pp inflation concentrated almost entirely in d1 glioma — the one class with a genuine distributional weakness across imaging sources. Leakage doesn't just make numbers look better; it selectively hides the failure modes that matter most. This is the finding I would most want to communicate to someone using these benchmarks.

**Wilson confidence intervals changed how I think about reporting results.** Reporting "papilloma recall = 65.7%" suggests a precise measurement. Reporting "papilloma recall = 65.7%, 95% CI [49.2%, 79.2%]" communicates that with 35 test examples, we genuinely do not know whether the model catches half or three-quarters of papillomas. For rare classes in medical imaging, this uncertainty quantification is arguably more important than the point estimate. This connects to the discussion of calibration and confidence from lecture — not just whether the model is confident, but whether *we* should be confident in our evaluation of the model.

**If I had two more weeks**, I would: (1) implement class-weighted loss or `WeightedRandomSampler` to address the severe imbalance, especially for data44-only classes; (2) obtain or infer true patient IDs for data44 to make the patient-level splitting more robust than filename heuristics; (3) train a lightweight baseline (logistic regression on flattened pixels, or a small CNN from scratch) to better contextualize how much of the performance comes from transfer learning vs. the task being inherently easy; and (4) develop the contamination analysis into a standalone short paper, since the controlled experiment design and per-source breakdown are novel contributions to this benchmark family.

**The most helpful feedback from my presentation** was the suggestion to quantify the leakage rather than just acknowledging it. The original plan was to note "patient-level leakage exists" as a limitation; the controlled experiment that isolates and measures it turned a limitation into a finding.